## Data Preparation
Load the supplied HLO ontology, resource catalog, question pairs and scored assessments. Development records are synthetic.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/hlo_recommender").is_dir())
os.chdir(ROOT)

from hlo_recommender.cli import parser, run
from hlo_recommender.io import read_jsonl, write_json, write_jsonl

SESSION_FILE = ROOT / "artifacts/notebooks/session.json"
SESSION = json.loads(SESSION_FILE.read_text()) if SESSION_FILE.exists() else {}


In [ ]:
source = json.loads((ROOT / "data/ontology/source.json").read_text())
manifest = json.loads((ROOT / "data/development/manifest.json").read_text())
resources = json.loads((ROOT / "data/development/resources.json").read_text())
{"ontology": source, "data": manifest, "resource_example": resources["resources"][0]}


### Collection
Read the existing files; reviewed exports may replace the source directory after expert labeling.

In [ ]:
from hlo_recommender.data import validate_events, validate_questions, validate_partitions
from hlo_recommender.io import grouped_split

SOURCE = ROOT / "data/development"
names = ("train", "validation", "test")
questions = [r for name in names for r in read_jsonl(SOURCE / f"questions.{name}.jsonl")]
assessments = [r for name in names for r in read_jsonl(SOURCE / f"assessments.{name}.jsonl")]
validate_questions(questions)
validate_events(assessments)
{"questions": len(questions), "assessments": len(assessments), "example": questions[0]}


### Partitioning
Use the project splitter to keep question families and learners in separate partitions.

In [ ]:
OUTPUT = ROOT / "artifacts/notebooks" / f"data-{uuid4().hex[:8]}"
question_splits = grouped_split(questions, "family_id", 42)
assessment_splits = grouped_split(assessments, "learner_id", 42)
summary = validate_partitions(question_splits)
for name in names:
    validate_events(assessment_splits[name])
    write_jsonl(OUTPUT / f"questions.{name}.jsonl", question_splits[name])
    write_jsonl(OUTPUT / f"assessments.{name}.jsonl", assessment_splits[name])
run(parser().parse_args(["data", "validate", "--directory", str(OUTPUT)]))
SESSION["data_directory"] = str(OUTPUT)
write_json(SESSION_FILE, SESSION)
summary


### Model Input
Inspect the existing serializer; reference queries and relevance labels stay outside the model input.

In [ ]:
from hlo_recommender.pipeline import Runtime
from hlo_recommender.models.mt5 import serialize_input

runtime = Runtime.from_file(ROOT / "configs/default.toml")
example = question_splits["train"][0]
schema = runtime.grounder.schema_text(example["question"])
print(serialize_input(example["question"], schema, example["context"]))
